# Finger Count — Full Dataset Training (Kaggle, GPU)

Trains on **all 5,615 real photos** of the Finger Count Cleaned Dataset, using the copy
already hosted on Kaggle — no download, no API token.

## Setup before you run anything

1. In the notebook's right-hand panel: **Add Input** -> search `Finger Count Cleaned Dataset`
   (by nayasakka88) -> **Add**. It appears under `/kaggle/input/`.
2. **Settings -> Accelerator -> GPU T4 x2** (or P100).
3. **Settings -> Internet -> On**. Kaggle asks for phone verification the first time.
   Needed only to `pip install ultralytics` if it isn't already present.
4. **Save Version -> Save & Run All** if you want it to run unattended and keep the output.

Output files land in `/kaggle/working/` and appear in the **Output** tab when the run ends.


In [ ]:
# 1. Environment check
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys, subprocess
try:
    import ultralytics, torch
except ImportError:
    print("installing ultralytics ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
    import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())


In [ ]:
# 2. Locate the dataset that Kaggle mounted under /kaggle/input
import glob, os

CLASSES = [str(i) for i in range(6)]
found = {}
for d in glob.glob("/kaggle/input/**/", recursive=True):
    name = os.path.basename(d.rstrip("/"))
    if name in CLASSES:
        n = len([f for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        found.setdefault(name, []).append((d, n))

if not found:
    raise SystemExit(
        "No class folders found under /kaggle/input.\n"
        "Use Add Input -> Finger Count Cleaned Dataset, then re-run this cell.\n"
        "Current contents: " + str(os.listdir('/kaggle/input'))
    )

print("class folders found:")
for c in CLASSES:
    if c in found:
        for d, n in found[c]:
            print(f"  {c}: {n} images   ({d})")
    else:
        print(f"  {c}: MISSING")

CLASS_DIR = os.path.dirname(sorted(found[CLASSES[0]])[0][0])
print("\nbase path:", CLASS_DIR)
total = sum(n for v in found.values() for _, n in v)
print("total images:", total)


In [ ]:
# 3. Eyeball 12 random images — confirm real hands, correct labels
import random
from PIL import Image
import matplotlib.pyplot as plt

paths = []
for c in CLASSES:
    d = os.path.join(CLASS_DIR, c)
    if os.path.isdir(d):
        fs = [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        paths += [(c, p) for p in random.sample(fs, min(2, len(fs)))]

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, (c, p) in zip(axes.ravel(), paths):
    ax.imshow(Image.open(p).convert("RGB")); ax.set_title(f"label = {c}"); ax.axis("off")
for ax in axes.ravel()[len(paths):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

print("sample resolutions:", [Image.open(p).size for _, p in paths[:6]])


In [ ]:
# 4. Build YOLO classification layout: /kaggle/working/cls/{train,val}/<class>
import os, random, shutil

DST = "/kaggle/working/cls"
random.seed(0)
shutil.rmtree(DST, ignore_errors=True)

for split in ("train", "val"):
    for c in CLASSES:
        os.makedirs(os.path.join(DST, split, c), exist_ok=True)

for c in CLASSES:
    src = os.path.join(CLASS_DIR, c)
    if not os.path.isdir(src):
        print(f"WARNING: class {c} folder not found — skipping")
        continue
    files = [os.path.join(src, f) for f in os.listdir(src) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    random.shuffle(files)
    n_val = max(1, int(len(files) * 0.15))
    for i, f in enumerate(files):
        split = "val" if i < n_val else "train"
        dst = os.path.join(DST, split, c, os.path.basename(f))
        if not os.path.exists(dst):
            os.symlink(f, dst)
    print(f"class {c}: {len(files)} -> {len(files)-n_val} train / {n_val} val")


In [ ]:
# 5. Train — roughly 12-20 min on a T4 for 40 epochs
from ultralytics import YOLO

model = YOLO("yolo11n-cls.pt")          # yolo11s-cls.pt = more accuracy, ~3x slower
model.train(
    data="/kaggle/working/cls",
    epochs=40,
    imgsz=224,
    batch=64,
    device=0,
    project="/kaggle/working/runs",
    name="finger_cls",
    exist_ok=True,
    plots=True,
    seed=0,
    patience=15,
)


In [ ]:
# 6. Honest evaluation
from ultralytics import YOLO
best = YOLO("/kaggle/working/runs/finger_cls/weights/best.pt")
m = best.val(data="/kaggle/working/cls", split="val", plots=True)
print(f"top-1 accuracy: {m.top1:.4f}")
print(f"top-5 accuracy: {m.top5:.4f}")


In [ ]:
# 7. Which counts are confused? Sample predictions per class
import os
from PIL import Image
import matplotlib.pyplot as plt

val_dir = "/kaggle/working/cls/val"
fig, axes = plt.subplots(2, 6, figsize=(20, 7))
i = 0
for c in CLASSES:
    d = os.path.join(val_dir, c)
    if not os.path.isdir(d): continue
    for f in [os.path.join(d, x) for x in os.listdir(d)][:2]:
        r = best.predict(f, verbose=False)[0]
        axes.ravel()[i].imshow(Image.open(f).convert("RGB"))
        axes.ravel()[i].set_title(f"true {c} / pred {r.names[int(r.probs.top1)]}")
        axes.ravel()[i].axis("off")
        i += 1
for ax in axes.ravel()[i:]: ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# 8. Collect the deliverables into /kaggle/working so they appear in the Output tab
import shutil, os

OUT = "/kaggle/working/deliverables"
os.makedirs(OUT, exist_ok=True)

src = "/kaggle/working/runs/finger_cls"
for f in ["weights/best.pt", "results.csv", "confusion_matrix_normalized.png",
          "confusion_matrix.png", "val_batch0_pred.jpg"]:
    p = os.path.join(src, f)
    if os.path.exists(p):
        shutil.copy(p, os.path.join(OUT, os.path.basename(f)))
        print("collected", f)

onnx_path = best.export(format="onnx", imgsz=224, opset=12)
shutil.copy(onnx_path, os.path.join(OUT, "finger_count.onnx"))
print("\nexported ONNX:", onnx_path)

print("\n--- deliverables ---")
!ls -lh {OUT}


## Getting the files out

The **Output** tab (right panel, or the version page after *Save Version*) lets you download
everything in `/kaggle/working/deliverables`:

- `best.pt` — trained weights, ~6 MB
- `finger_count.onnx` — for the website demo
- `results.csv` — per-epoch metrics
- `confusion_matrix_normalized.png` — which counts get mixed up

Attach `best.pt`, `results.csv` and the confusion matrix to the chat.

## Kaggle gotchas

| Symptom | Fix |
|---|---|
| `No class folders found under /kaggle/input` | You forgot **Add Input** — add the dataset, re-run cell 2 |
| `pip install` fails with network error | **Settings -> Internet -> On** (needs phone verification) |
| Session dies at ~9 hours | Kaggle caps sessions; 40 epochs won't come close |
| `CUDA out of memory` | batch 64 -> 32, or imgsz 224 -> 160 |
| Output tab shows nothing | Run cell 8, then **Save Version** |
| Only 5 class folders appear | The `5` folder is named differently — tell me what cell 2 printed |

## Optional: detector instead of classifier

The dataset has no bounding boxes, so a detector needs pseudo-boxes (whole image = one box).
That only teaches "a hand filling the frame", which is why the classifier is the honest choice.
If you want it anyway, see the last cell of the Colab version of this notebook.
